# Silver Project Validation

## Purpose

This notebook validates the Silver DPWH project foundation introduced by:
 
- `02-silver`.silver_dpwh_project_component
- `02-silver`.silver_project
 
The validation confirms that source rows remain preserved, project-level accounting remains consistent, project budgets are not double counted, source lineage remains traceable, and unresolved source-quality findings remain visible rather than being silently corrected.
 
Invalid values should become review findings rather than automatic transformations.

## 1. Validate upstream safety

This validation confirms that the Silver project foundation was created from a valid Bronze source.

The Bronze DPWH source must contain data and the published Silver tables must also contain records.

Expected result:

- Bronze DPWH source contains rows
- Silver component table contains rows
- Tables are available for downstream processing

Failure of these checks represents a structural issue and should stop processing.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW latest_dpwh_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY
                COALESCE(completed_at, started_at) DESC,
                started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'dpwh_projects'
      AND status IN (
          'STARTED',
          'SUCCESS',
          'FAILED',
          'SKIPPED_IDEMPOTENT'
      )
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(
        status IN (
            'SUCCESS',
            'SKIPPED_IDEMPOTENT'
        )
    ) = 1
    AND COUNT_IF(
        snapshot_id IS NOT NULL
        AND TRIM(snapshot_id) <> ''
    ) = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'STOP: latest DPWH Bronze load is not safe'
)
FROM latest_dpwh_load;

CREATE OR REPLACE TEMPORARY VIEW latest_dpwh_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT
        *,
        DENSE_RANK() OVER (
            ORDER BY
                run_ts DESC,
                run_id DESC
        ) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'dpwh_projects'
      AND snapshot_id = (
          SELECT snapshot_id
          FROM latest_dpwh_load
      )
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(
        action = 'stop'
        AND status IN (
            'FAIL',
            'ERROR'
        )
    ) = 0,
    'STOP: latest DPWH Bronze validation is unsafe'
)
FROM latest_dpwh_validation;

SELECT
    table_name,
    status,
    snapshot_id,
    rows_loaded
FROM latest_dpwh_load;

## 2. Validate component row preservation

The component table must preserve the Bronze source row grain.

One component-level Silver row should exist for every selected Bronze DPWH source row.

The goal of this validation is to ensure that rows were not silently removed, duplicated, or collapsed during parsing and standardization.

Expected result:

- Bronze row count equals Silver component row count
- Row difference equals zero

Any mismatch is treated as a stop-condition failure.

In [ ]:
%sql
WITH row_counts AS (
    SELECT
        (
            SELECT COUNT(*)
            FROM `01-bronze`.dpwh_projects
        ) AS bronze_rows,

        (
            SELECT COUNT(*)
            FROM `02-silver`.silver_dpwh_project_component
        ) AS silver_component_rows
)

SELECT
    bronze_rows,
    silver_component_rows,
    silver_component_rows - bronze_rows AS row_difference,

    CASE
        WHEN bronze_rows = silver_component_rows
            THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status,

    CASE
        WHEN bronze_rows = silver_component_rows
            THEN 'proceed'
        ELSE 'stop'
    END AS validation_action

FROM row_counts;

In [ ]:
%sql
WITH row_counts AS (
    SELECT
        (
            SELECT COUNT(*)
            FROM `01-bronze`.dpwh_projects
        ) AS bronze_rows,

        (
            SELECT COUNT(*)
            FROM `02-silver`.silver_dpwh_project_component
        ) AS silver_component_rows
)

SELECT ASSERT_TRUE(
    bronze_rows = silver_component_rows,
    'STOP: Bronze-to-Silver component row reconciliation failed'
)
FROM row_counts;

## 3. Validate project-row reconciliation

The project table consolidates component-level evidence into one project-level record.

This validation confirms that all component rows remain fully accounted for after consolidation.

Expected result:

- SUM(component_count) equals component-table row count
- Project grain remains unique
- No project rows are duplicated

The component table remains the source evidence used to explain every project-level result.

In [ ]:
%sql
WITH component_accounting AS (
    SELECT
        (
            SELECT COUNT(*)
            FROM `02-silver`.silver_dpwh_project_component
        ) AS component_rows,

        (
            SELECT SUM(component_count)
            FROM `02-silver`.silver_project
        ) AS accounted_component_rows
)

SELECT
    component_rows,
    accounted_component_rows,
    accounted_component_rows - component_rows AS accounting_difference,

    CASE
        WHEN component_rows = accounted_component_rows
            THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status,

    CASE
        WHEN component_rows = accounted_component_rows
            THEN 'proceed'
        ELSE 'stop'
    END AS validation_action

FROM component_accounting;

In [ ]:
%sql
WITH component_accounting AS (
    SELECT
        (
            SELECT COUNT(*)
            FROM `02-silver`.silver_dpwh_project_component
        ) AS component_rows,

        (
            SELECT SUM(component_count)
            FROM `02-silver`.silver_project
        ) AS accounted_component_rows
)

SELECT ASSERT_TRUE(
    component_rows = accounted_component_rows,
    'STOP: Silver project component accounting failed'
)
FROM component_accounting;

In [ ]:
%sql
WITH component_accounting AS (
    SELECT
        (
            SELECT COUNT(*)
            FROM `02-silver`.silver_dpwh_project_component
        ) AS component_rows,

        (
            SELECT SUM(component_count)
            FROM `02-silver`.silver_project
        ) AS accounted_component_rows
)

SELECT ASSERT_TRUE(
    component_rows = accounted_component_rows,
    'STOP: Silver project component accounting failed'
)
FROM component_accounting;

## 4. Validate budget resolution

Reported project budgets must not be double counted.

Repeated identical values may resolve to one canonical project budget, while conflicting values remain unresolved until an approved business rule exists.

Expected result:

- Resolved budgets remain explainable
- Conflicting budgets remain visible
- Unsupported project totals are never created automatically

Budget conflicts are reported as review findings rather than silently corrected.

In [ ]:
%sql
SELECT
    budget_resolution_status,
    COUNT(contract_id) AS project_count,

    ROUND(
        100.0 * COUNT(contract_id)
        / SUM(COUNT(contract_id)) OVER (),
        2
    ) AS project_percentage

FROM `02-silver`.silver_project
GROUP BY budget_resolution_status
ORDER BY project_count DESC;


In [ ]:
%sql
WITH invalid_resolutions AS (
    SELECT
        contract_id,
        budget_resolution_status,
        distinct_budget_value_count,
        reported_budget_pesos
    FROM `02-silver`.silver_project
    WHERE
        (
            distinct_budget_value_count = 1
            AND reported_budget_pesos IS NULL
        )
        OR
        (
            distinct_budget_value_count > 1
            AND reported_budget_pesos IS NOT NULL
        )
)

SELECT ASSERT_TRUE(
    COUNT(contract_id) = 0,
    'STOP: project budget resolution violates the double-count protection rule'
)
FROM invalid_resolutions;


## 5. Report retained source-quality findings

These checks report malformed, missing, conflicting, or unmapped source values that remain visible in Silver.

The findings are FLAGS rather than automatic corrections. Silver preserves the original source evidence instead of replacing questionable values with guessed values.

In [ ]:
%sql
SELECT
    coordinate_status,
    COUNT(contract_id) AS component_count,

    ROUND(
        100.0 * COUNT(contract_id)
        / SUM(COUNT(contract_id)) OVER (),
        2
    ) AS component_percentage,

    CASE
        WHEN coordinate_status = 'VALID_PAIR'
            THEN 'PASS'
        ELSE 'FLAG'
    END AS validation_status,

    CASE
        WHEN coordinate_status = 'VALID_PAIR'
            THEN 'proceed'
        ELSE 'review'
    END AS validation_action

FROM `02-silver`.silver_dpwh_project_component
GROUP BY coordinate_status
ORDER BY component_count DESC;


In [ ]:
%sql
SELECT
    COUNT(contract_id) AS total_components,

    COUNT(
        CASE
            WHEN reported_budget_raw IS NOT NULL
             AND reported_budget_pesos IS NULL
                THEN contract_id
        END
    ) AS budget_parse_failures,

    COUNT(
        CASE
            WHEN reported_budget_pesos < 0
                THEN contract_id
        END
    ) AS negative_budget_rows,

    CASE
        WHEN COUNT(
            CASE
                WHEN reported_budget_raw IS NOT NULL
                 AND reported_budget_pesos IS NULL
                    THEN contract_id
            END
        ) > 0
            THEN 'FLAG'
        ELSE 'PASS'
    END AS validation_status

FROM `02-silver`.silver_dpwh_project_component;

In [ ]:
%sql
SELECT
    COUNT(contract_id) AS total_components,

    COUNT(
        CASE
            WHEN reported_budget_raw IS NOT NULL
             AND reported_budget_pesos IS NULL
                THEN contract_id
        END
    ) AS budget_parse_failures,

    COUNT(
        CASE
            WHEN reported_budget_pesos < 0
                THEN contract_id
        END
    ) AS negative_budget_rows,

    CASE
        WHEN COUNT(
            CASE
                WHEN reported_budget_raw IS NOT NULL
                 AND reported_budget_pesos IS NULL
                    THEN contract_id
            END
        ) > 0
            THEN 'FLAG'
        ELSE 'PASS'
    END AS validation_status

FROM `02-silver`.silver_dpwh_project_component;

In [ ]:
%sql
SELECT
    COUNT(contract_id) AS total_components,

    COUNT(
        CASE
            WHEN progress_raw IS NOT NULL
             AND physical_progress_pct IS NULL
                THEN contract_id
        END
    ) AS progress_parse_failures,

    COUNT(
        CASE
            WHEN physical_progress_pct < 0
              OR physical_progress_pct > 100
                THEN contract_id
        END
    ) AS progress_outside_valid_range,

    CASE
        WHEN COUNT(
            CASE
                WHEN progress_raw IS NOT NULL
                 AND physical_progress_pct IS NULL
                    THEN contract_id
            END
        ) > 0
          OR COUNT(
            CASE
                WHEN physical_progress_pct < 0
                  OR physical_progress_pct > 100
                    THEN contract_id
            END
        ) > 0
            THEN 'FLAG'
        ELSE 'PASS'
    END AS validation_status

FROM `02-silver`.silver_dpwh_project_component;


In [ ]:
%sql
SELECT
    COUNT(
        CASE
            WHEN start_date_raw IS NOT NULL
             AND start_date_parsed IS NULL
                THEN contract_id
        END
    ) AS start_date_parse_failures,

    COUNT(
        CASE
            WHEN completion_date_raw IS NOT NULL
             AND completion_date_parsed IS NULL
                THEN contract_id
        END
    ) AS completion_date_parse_failures,

    CASE
        WHEN COUNT(
            CASE
                WHEN start_date_raw IS NOT NULL
                 AND start_date_parsed IS NULL
                    THEN contract_id
            END
        ) > 0
          OR COUNT(
            CASE
                WHEN completion_date_raw IS NOT NULL
                 AND completion_date_parsed IS NULL
                    THEN contract_id
            END
        ) > 0
            THEN 'FLAG'
        ELSE 'PASS'
    END AS validation_status

FROM `02-silver`.silver_dpwh_project_component;


In [ ]:
%sql
SELECT
    COUNT(contract_id) AS total_projects,

    COUNT(
        CASE
            WHEN description_resolution_status = 'CONFLICT'
                THEN contract_id
        END
    ) AS description_conflicts,

    COUNT(
        CASE
            WHEN category_resolution_status = 'CONFLICT'
                THEN contract_id
        END
    ) AS category_conflicts,

    COUNT(
        CASE
            WHEN status_resolution_status = 'CONFLICT'
                THEN contract_id
        END
    ) AS status_conflicts,

    COUNT(
        CASE
            WHEN progress_resolution_status = 'CONFLICT'
                THEN contract_id
        END
    ) AS progress_conflicts,

    COUNT(
        CASE
            WHEN start_date_resolution_status = 'CONFLICT'
                THEN contract_id
        END
    ) AS start_date_conflicts,

    COUNT(
        CASE
            WHEN completion_date_resolution_status = 'CONFLICT'
                THEN contract_id
        END
    ) AS completion_date_conflicts,

    COUNT(
        CASE
            WHEN budget_resolution_status = 'CONFLICT'
                THEN contract_id
        END
    ) AS budget_conflicts,

    CASE
        WHEN COUNT(
            CASE
                WHEN description_resolution_status = 'CONFLICT'
                  OR category_resolution_status = 'CONFLICT'
                  OR status_resolution_status = 'CONFLICT'
                  OR progress_resolution_status = 'CONFLICT'
                  OR start_date_resolution_status = 'CONFLICT'
                  OR completion_date_resolution_status = 'CONFLICT'
                  OR budget_resolution_status = 'CONFLICT'
                    THEN contract_id
            END
        ) > 0
            THEN 'FLAG'
        ELSE 'PASS'
    END AS validation_status

FROM `02-silver`.silver_project;


## 6. Validate lineage

Every Silver output must remain traceable back to the originating Bronze source.

Source snapshot identifiers and ingest lineage must remain available so that project records can be reproduced and audited.

Expected result:

- Source snapshot lineage exists
- Bronze ingest lineage exists
- Downstream project outputs remain traceable

Missing lineage is treated as a stop-condition failure.

In [ ]:
%sql
SELECT
    COUNT(contract_id) AS total_components,

    COUNT(
        CASE
            WHEN _source_snapshot_id IS NULL
              OR TRIM(_source_snapshot_id) = ''
                THEN contract_id
        END
    ) AS missing_snapshot_lineage,

    COUNT(
        CASE
            WHEN _ingest_run_id IS NULL
              OR TRIM(_ingest_run_id) = ''
                THEN contract_id
        END
    ) AS missing_ingest_lineage

FROM `02-silver`.silver_dpwh_project_component;


In [ ]:
%sql
WITH missing_lineage AS (
    SELECT
        COUNT(
            CASE
                WHEN _source_snapshot_id IS NULL
                  OR TRIM(_source_snapshot_id) = ''
                  OR _ingest_run_id IS NULL
                  OR TRIM(_ingest_run_id) = ''
                    THEN contract_id
            END
        ) AS failed_rows
    FROM `02-silver`.silver_dpwh_project_component
)

SELECT ASSERT_TRUE(
    failed_rows = 0,
    'STOP: mandatory source lineage is missing from Silver components'
)
FROM missing_lineage;


## 7. Persist Silver project validation evidence

This section consolidates the project-foundation validation metrics into one reusable result set.

The result rows are written to `04-validation`.silver_dq_results before the blocking gate is enforced. This preserves evidence from the validation attempt even when a blocking check fails.

The validation run identity is deterministic for the selected source snapshot, Silver project run, and validation-rule version. Exact reruns update the existing evidence instead of appending duplicate logical checks.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_project_validation_metrics AS
SELECT
    (
        SELECT COUNT(*)
        FROM `01-bronze`.dpwh_projects
    ) AS bronze_rows,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_dpwh_project_component
    ) AS component_rows,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_project
    ) AS project_rows,

    (
        SELECT COALESCE(SUM(component_count), 0)
        FROM `02-silver`.silver_project
    ) AS accounted_component_rows,

    (
        SELECT COUNT(*)
        FROM (
            SELECT
                contract_id
            FROM `02-silver`.silver_project
            GROUP BY contract_id
            HAVING COUNT(*) > 1
        )
    ) AS duplicate_project_keys,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_dpwh_project_component
        WHERE contract_id IS NULL
           OR TRIM(contract_id) = ''
    ) AS missing_contract_ids,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_dpwh_project_component
        WHERE _source_snapshot_id IS NULL
           OR TRIM(_source_snapshot_id) = ''
           OR _ingest_run_id IS NULL
           OR TRIM(_ingest_run_id) = ''
    ) AS missing_component_lineage,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_project
        WHERE
            (
                distinct_budget_value_count = 1
                AND reported_budget_pesos IS NULL
            )
            OR
            (
                distinct_budget_value_count > 1
                AND reported_budget_pesos IS NOT NULL
            )
    ) AS invalid_budget_resolutions,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_dpwh_project_component
        WHERE coordinate_status <> 'VALID_PAIR'
    ) AS coordinate_exception_rows,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_dpwh_project_component
        WHERE reported_budget_raw IS NOT NULL
          AND reported_budget_pesos IS NULL
    ) AS budget_parse_failure_rows,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_dpwh_project_component
        WHERE physical_progress_pct < 0
           OR physical_progress_pct > 100
    ) AS invalid_progress_rows,

    (
        SELECT COUNT(*)
        FROM `02-silver`.silver_project
        WHERE description_resolution_status = 'CONFLICT'
           OR category_resolution_status = 'CONFLICT'
           OR status_resolution_status = 'CONFLICT'
           OR progress_resolution_status = 'CONFLICT'
           OR start_date_resolution_status = 'CONFLICT'
           OR completion_date_resolution_status = 'CONFLICT'
           OR budget_resolution_status = 'CONFLICT'
    ) AS project_conflict_rows;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_project_validation_context AS
SELECT
    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(load.snapshot_id, ''),
            COALESCE(project_run.pipeline_run_id, ''),
            'silver_project_validation_v1'
        ),
        256
    ) AS validation_run_id,

    project_run.pipeline_run_id,
    load.snapshot_id AS source_snapshot_id,
    'silver_project_validation_v1' AS rule_version,
    CURRENT_TIMESTAMP() AS run_ts

FROM latest_dpwh_load AS load

CROSS JOIN (
    SELECT
        MAX(run_id) AS pipeline_run_id
    FROM `02-silver`.silver_project
) AS project_run;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_project_checks AS
WITH check_rows AS (
    SELECT
        'silver_dpwh_project_component' AS table_name,
        'Completeness' AS data_quality_attribute,
        'Bronze DPWH rows reconcile to Silver component rows'
            AS data_quality_check,
        ABS(component_rows - bronze_rows) AS failed_rows,
        bronze_rows AS total_rows,
        CASE
            WHEN bronze_rows = 0 THEN NULL
            ELSE
                100.0 * ABS(component_rows - bronze_rows)
                / bronze_rows
        END AS percentage,
        CASE
            WHEN component_rows = bronze_rows THEN 'PASS'
            ELSE 'FAIL'
        END AS status,
        'stop' AS action,
        CONCAT(
            'Bronze rows=', bronze_rows,
            '; component rows=', component_rows,
            '; difference=', component_rows - bronze_rows
        ) AS details
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_project',
        'Completeness',
        'Project component counts reconcile to component-table rows',
        ABS(accounted_component_rows - component_rows),
        component_rows,
        CASE
            WHEN component_rows = 0 THEN NULL
            ELSE
                100.0
                * ABS(accounted_component_rows - component_rows)
                / component_rows
        END,
        CASE
            WHEN accounted_component_rows = component_rows THEN 'PASS'
            ELSE 'FAIL'
        END,
        'stop',
        CONCAT(
            'Component rows=', component_rows,
            '; accounted rows=', accounted_component_rows,
            '; difference=',
            accounted_component_rows - component_rows
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_project',
        'Uniqueness',
        'Project Contract ID grain is unique',
        duplicate_project_keys,
        project_rows,
        CASE
            WHEN project_rows = 0 THEN NULL
            ELSE 100.0 * duplicate_project_keys / project_rows
        END,
        CASE
            WHEN duplicate_project_keys = 0 THEN 'PASS'
            ELSE 'FAIL'
        END,
        'stop',
        CONCAT(
            'Duplicate project keys=',
            duplicate_project_keys
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_dpwh_project_component',
        'Validity',
        'Contract ID is present before project consolidation',
        missing_contract_ids,
        component_rows,
        CASE
            WHEN component_rows = 0 THEN NULL
            ELSE 100.0 * missing_contract_ids / component_rows
        END,
        CASE
            WHEN missing_contract_ids = 0 THEN 'PASS'
            ELSE 'FAIL'
        END,
        'stop',
        CONCAT(
            'Missing or blank Contract IDs=',
            missing_contract_ids
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_dpwh_project_component',
        'Auditability',
        'Mandatory component source lineage is populated',
        missing_component_lineage,
        component_rows,
        CASE
            WHEN component_rows = 0 THEN NULL
            ELSE 100.0 * missing_component_lineage / component_rows
        END,
        CASE
            WHEN missing_component_lineage = 0 THEN 'PASS'
            ELSE 'FAIL'
        END,
        'stop',
        CONCAT(
            'Rows missing mandatory lineage=',
            missing_component_lineage
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_project',
        'Accuracy',
        'Project budgets follow the conservative resolution contract',
        invalid_budget_resolutions,
        project_rows,
        CASE
            WHEN project_rows = 0 THEN NULL
            ELSE 100.0 * invalid_budget_resolutions / project_rows
        END,
        CASE
            WHEN invalid_budget_resolutions = 0 THEN 'PASS'
            ELSE 'FAIL'
        END,
        'stop',
        CONCAT(
            'Invalid budget resolutions=',
            invalid_budget_resolutions
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_dpwh_project_component',
        'Completeness',
        'Coordinate exceptions remain visible',
        coordinate_exception_rows,
        component_rows,
        CASE
            WHEN component_rows = 0 THEN NULL
            ELSE 100.0 * coordinate_exception_rows / component_rows
        END,
        CASE
            WHEN coordinate_exception_rows = 0 THEN 'PASS'
            ELSE 'FLAG'
        END,
        'flag',
        CONCAT(
            'Missing, partial, or unparseable coordinate rows=',
            coordinate_exception_rows
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_dpwh_project_component',
        'Validity',
        'Reported budget parse failures remain visible',
        budget_parse_failure_rows,
        component_rows,
        CASE
            WHEN component_rows = 0 THEN NULL
            ELSE 100.0 * budget_parse_failure_rows / component_rows
        END,
        CASE
            WHEN budget_parse_failure_rows = 0 THEN 'PASS'
            ELSE 'FLAG'
        END,
        'flag',
        CONCAT(
            'Budget parse failure rows=',
            budget_parse_failure_rows
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_dpwh_project_component',
        'Validity',
        'Physical progress remains within zero to one hundred',
        invalid_progress_rows,
        component_rows,
        CASE
            WHEN component_rows = 0 THEN NULL
            ELSE 100.0 * invalid_progress_rows / component_rows
        END,
        CASE
            WHEN invalid_progress_rows = 0 THEN 'PASS'
            ELSE 'FLAG'
        END,
        'flag',
        CONCAT(
            'Progress values outside zero to one hundred=',
            invalid_progress_rows
        )
    FROM silver_project_validation_metrics

    UNION ALL

    SELECT
        'silver_project',
        'Consistency',
        'Project-level attribute conflicts remain visible',
        project_conflict_rows,
        project_rows,
        CASE
            WHEN project_rows = 0 THEN NULL
            ELSE 100.0 * project_conflict_rows / project_rows
        END,
        CASE
            WHEN project_conflict_rows = 0 THEN 'PASS'
            ELSE 'FLAG'
        END,
        'flag',
        CONCAT(
            'Projects with at least one conflicting attribute=',
            project_conflict_rows
        )
    FROM silver_project_validation_metrics
)

SELECT
    context.validation_run_id,
    context.pipeline_run_id,
    'silver' AS layer,
    checks.table_name,
    context.source_snapshot_id,
    context.rule_version,
    checks.data_quality_attribute,
    checks.data_quality_check,
    CAST(checks.failed_rows AS BIGINT) AS failed_rows,
    CAST(checks.total_rows AS BIGINT) AS total_rows,
    CAST(checks.percentage AS DOUBLE) AS percentage,
    checks.status,
    checks.action,
    checks.details,
    context.run_ts

FROM check_rows AS checks
CROSS JOIN silver_project_validation_context AS context;

# Summary

Validated the Silver DPWH project foundation at both component and project grains.

The validation confirms whether:

- Bronze DPWH rows reconcile exactly to component-level Silver rows
- Component counts reconcile exactly to project-level accounting
- Project Contract IDs are unique at the documented grain
- Repeated budget values are not blindly summed
- Conflicting project-level values remain visible
- Invalid budgets, progress values, dates, and coordinates remain visible as FLAG findings
- Mandatory source snapshot and Bronze ingest lineage remain traceable

Source-quality findings are retained for review rather than silently corrected or deleted.


In [ ]:
%sql
MERGE INTO `04-validation`.silver_dq_results AS target
USING silver_project_checks AS source
    ON target.validation_run_id = source.validation_run_id
   AND target.table_name = source.table_name
   AND target.data_quality_check = source.data_quality_check

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT_IF(
        status = 'FAIL'
        AND action = 'stop'
    ) = 0,
    'STOP: Silver project validation failed'
)
FROM silver_project_checks;

SELECT
    status,
    action,
    COUNT(*) AS check_count
FROM silver_project_checks
GROUP BY
    status,
    action
ORDER BY
    status,
    action;